# Landsat 1 / MSS (Multispectral Scanner)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=landsat/landsat-1-mss). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1

Level 1 (Collection 2): geometrically corrected, calibrated digital numbers with scale factors in the metadata. MSS has no surface-reflectance Level-2 product.

Searches the Microsoft Planetary Computer STAC catalogue (no account needed) for Landsat Collection 2 Level-1 MSS scenes and plots the near-infrared and red bands. MSS coverage over Europe is patchy, so the search covers Italy and the whole mission life.

In [ ]:
%pip install -q pystac-client planetary-computer rioxarray matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rioxarray

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


def open_overview(href, level=2):
    """Open a COG at reduced resolution (fast), falling back to full resolution."""
    try:
        return rioxarray.open_rasterio(href, overview_level=level, masked=True).squeeze()
    except Exception:
        return rioxarray.open_rasterio(href, masked=True).squeeze()

search = catalog.search(
    collections=["landsat-c2-l1"],
    bbox=[6.0, 36.0, 19.0, 47.0],       # Italy - MSS coverage over Europe is patchy
    datetime="1972-01-01/1978-12-31",
    query={"platform": {"eq": "landsat-1"}},
)
items = search.item_collection()
if not items:
    raise SystemExit("No scenes found - try another date range or area")
item = min(items, key=lambda i: i.properties.get("eo:cloud_cover", 100))
print(len(items), "scenes found; using", item.id, item.datetime)
print("assets:", sorted(item.assets))

# MSS has no blue band: show the near-infrared band (nir08) and the red band
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, key in zip(axes, ["nir08", "red"]):
    open_overview(item.assets[key].href, level=3).plot(ax=ax, cmap="gray", robust=True, add_colorbar=False)
    ax.set_title(key)
plt.suptitle(item.id)
plt.show()